## P33_37b
- Loss landscapes over image *batches*: N in [1..128], independent random draws, 2 direction seeds x 4 image seeds
- Low-res: all 11 landscapes on one 3x4 figure per (N, img seed, dir seed). Hi-res: npy + texture + contour per landscape.

In [1]:
import json, gc, io, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.figure import Figure                      # OO figures: never registered with the notebook backend -> no RAM creep
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.transforms import Bbox
from torchvision.models.resnet import ResNet, BasicBlock
import torchvision.datasets as dsets, torchvision.transforms as T
from PIL import Image
from IPython.display import display

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'serif'

RUNS   = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/aug_17_run")
DATA   = Path("/home/stephen/imagenet")
OUT    = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin/P33_landscapes_v6")
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
print(torch.__version__, device)

2.12.1+cu130 cuda


In [2]:
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

def weighted_layers(model):
    '''(name, module) for convs + fc in forward order, excluding 1x1 shortcuts.'''
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear)) and "downsample" not in n]

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_ds = dsets.ImageFolder(DATA / "ILSVRC/Data/CLS-LOC/val",
    T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]))

def load_image(idx):
    x, y = val_ds[idx]
    return x.unsqueeze(0).to(device), torch.tensor([y], device=device)

crit = nn.CrossEntropyLoss()

@torch.no_grad()
def correct_ans_conf(model, x, y):
    return F.softmax(model(x), dim=1)[0, y.item()].item()

def layer_grad(model, layer, x, y):
    '''dL/dw for one layer at the current weights.'''
    model.zero_grad(set_to_none=True)
    crit(model(x), y).backward()
    g = layer.weight.grad.detach().flatten().clone()
    model.zero_grad(set_to_none=True)
    return g

@torch.no_grad()
def sweep_weight(model, layer, flat_idx, x, y, span, n):
    '''Set one weight to each value in linspace(-span, span, n), record P(correct), restore. Returns (vals, res, w0).'''
    w = layer.weight.view(-1)
    w0 = w[flat_idx].item()
    vals = np.linspace(-span, span, n)
    res = np.empty(n)
    for i, v in enumerate(vals):
        w[flat_idx] = v
        res[i] = correct_ans_conf(model, x, y)
    w[flat_idx] = w0
    return vals, res, w0

def theta_latex(coord, layer_num):
    inner = r',\,'.join(str(int(c)) for c in coord)
    return r'$\theta_{(' + inner + r')}^{(' + str(layer_num) + r')}$'

## Setup — run these cells for *either* section

Config + all helpers. Both sections depend on everything up to the **Section 1** header.

In [3]:
# ---- config ----
EXTENT        = 1.5      # global alpha/beta range for the low-res grid; hi-res can override per config
LOSS_CAP      = 32.0    # clip for plots + textures only; .npy / .npz store raw loss
GRID_LOW      = 32
GRID_HIGH     = 192
N_IMAGES_LIST = [1]
N_IMG_SEEDS   = 64        # independent image draws per N
IMG_SEED_BASE = 51       # image seeds IMG_SEED_BASE .. +N_IMG_SEEDS-1
N_DIRECTIONS  = 4        # direction pairs; seeds DIR_SEED_BASE .. +N_DIRECTIONS-1, shared across configs and batches
DIR_SEED_BASE = 24
SKIP_EXISTING = True     # resume: skip any figure/landscape whose cache file already exists
CMAP          = "viridis"
LOWRES_OUT    = OUT / "lowres"

# cfg name -> (checkpoint, indices into weighted_layers). weighted_layers includes stem conv (0) and fc (-1).
LANDSCAPES = {
    "plain8_first3":   ("plain8",   [0, 1, 2]),
    "plain8_last3":    ("plain8",   [-3, -2, -1]),
    "plain26_first4":  ("plain26",  [0, 1, 2, 3]),
    "plain74_first4":  ("plain74",  [0, 1, 2, 3]),
    "plain74_last4":   ("plain74",  [-4, -3, -2, -1]),
    "resnet74_first4": ("resnet74", [0, 1, 2, 3]),
    "resnet74_last4":  ("resnet74", [-4, -3, -2, -1]),
    "plain74_first8":  ("plain74",  [0, 1, 2, 3, 4, 5, 6, 7]),
    "resnet74_first8": ("resnet74", [0, 1, 2, 3, 4, 5, 6, 7]),
    "plain74_last8":   ("plain74",  [-8, -7, -6, -5, -4, -3, -2, -1]),
    "resnet74_last8":  ("resnet74", [-8, -7, -6, -5, -4, -3, -2, -1]),
}
assert len(LANDSCAPES) <= 11, "the 3x4 figure reserves one slot for the batch thumbnails"

IMG_SEEDS = list(range(IMG_SEED_BASE, IMG_SEED_BASE + N_IMG_SEEDS))
DIR_SEEDS = list(range(DIR_SEED_BASE, DIR_SEED_BASE + N_DIRECTIONS))
LOWRES_OUT.mkdir(parents=True, exist_ok=True)
print("image seeds:", IMG_SEEDS, "  dir seeds:", DIR_SEEDS)
print("figures in low-res sweep:", len(N_IMAGES_LIST) * len(IMG_SEEDS) * len(DIR_SEEDS))


image seeds: [51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114]   dir seeds: [24, 25, 26, 27]
figures in low-res sweep: 256


In [4]:
# ---- batches + models ----
def batch_indices(n, img_seed):
    """Independent draw of n val indices for each (n, img_seed) pair. Deterministic."""
    return np.random.default_rng([int(img_seed), int(n)]).choice(len(val_ds), size=n, replace=False)

def load_batch(idxs):
    xs, ys = zip(*(val_ds[int(i)] for i in idxs))
    return torch.stack(xs).to(device), torch.tensor(ys, device=device)

def denorm(x):
    """CHW normalized tensor -> HWC numpy in [0, 1]."""
    img = x.cpu() * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

def montage(idxs, max_show=16, stride=4):
    """Square grid of up to max_show thumbnails (224/stride px each) for the 12th panel."""
    idxs = list(idxs)[:max_show]
    k = int(np.ceil(np.sqrt(len(idxs))))
    t = 224 // stride
    canvas = np.ones((k * t, k * t, 3))
    for i, idx in enumerate(idxs):
        r, c = divmod(i, k)
        canvas[r*t:(r+1)*t, c*t:(c+1)*t] = denorm(val_ds[int(idx)][0])[::stride, ::stride]
    return canvas

_models = {}
def get_model(mn):
    """Checkpoints stay resident on the GPU once loaded (4 models ~ small)."""
    if mn not in _models:
        _models[mn] = load_model(mn)
    return _models[mn]

def free_models():
    _models.clear(); gc.collect(); torch.cuda.empty_cache()


In [5]:
# ---- directions + surface (unchanged from P33; loss_surface works on a batch since crit averages) ----
def filter_normalize_(d, w):
    """In-place: rescale each filter of d (row for Linear) to match w's filter norm."""
    d2, w2 = d.flatten(1), w.flatten(1)
    d2.mul_(w2.norm(dim=1, keepdim=True) / (d2.norm(dim=1, keepdim=True) + 1e-10))
    return d

def make_directions(model, layer_idxs, seed):
    """Two filter-normalized random directions over ONLY the selected layers (Li et al. 2018).
    Same seed -> same Gaussian draw for any model with matching layer shapes (e.g. plain74 vs resnet74)."""
    g = torch.Generator().manual_seed(seed)
    layers = weighted_layers(model)
    dirs = []
    for _ in range(2):
        d = {}
        for i in layer_idxs:
            name, m = layers[i]
            r = torch.randn(m.weight.shape, generator=g).to(device)
            d[name] = filter_normalize_(r, m.weight.data)
        dirs.append(d)
    return dirs  # [delta, eta]

@torch.no_grad()
def loss_surface(model, delta, eta, x, y, grid_n, extent=EXTENT):
    """Z[i, j] = mean-CE loss over the batch at alpha = lin[j], beta = lin[i].
    Weights are restored on exit (even on KeyboardInterrupt)."""
    lin = np.linspace(-extent, extent, grid_n)
    layers = dict(weighted_layers(model))
    orig = {n: layers[n].weight.data.clone() for n in delta}
    Z = np.empty((grid_n, grid_n))
    try:
        for i, b in enumerate(lin):
            for j, a in enumerate(lin):
                for n in delta:
                    layers[n].weight.data.copy_(orig[n]).add_(delta[n], alpha=a).add_(eta[n], alpha=b)
                Z[i, j] = crit(model(x), y).item()
    finally:
        for n in delta:
            layers[n].weight.data.copy_(orig[n])
    return Z


In [6]:
# ---- plotting ----
def draw_contour(ax, Z, extent, loss_cap, levels=25, lw=0.3):
    lin = np.linspace(-extent, extent, Z.shape[0]); A, B = np.meshgrid(lin, lin)
    Zc = np.clip(Z, None, loss_cap)
    cs = ax.contourf(A, B, Zc, levels=levels, cmap=CMAP)
    ax.contour(A, B, Zc, levels=levels, colors="k", linewidths=lw, alpha=0.4)
    ax.plot(0, 0, "r+", ms=9)
    return cs

def grid_fig(Zs, idxs, n, img_seed, dir_seed, extent=EXTENT, loss_cap=None, panel=3.3):
    """3x4 figure: the 11 landscapes (LANDSCAPES order) + batch thumbnails in the last slot.
    Each panel has its own color scale."""
    loss_cap = LOSS_CAP if loss_cap is None else loss_cap
    fig = Figure(figsize=(4 * panel, 3 * panel * 0.98)); FigureCanvasAgg(fig)
    axes = fig.subplots(3, 4).ravel()
    for ax, cfg in zip(axes, LANDSCAPES):
        Z = Zs[cfg]
        draw_contour(ax, Z, extent, loss_cap, lw=0.25)
        ax.set_title(f"{cfg}   [{Z.min():.2f}, {Z.max():.1f}]", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[len(LANDSCAPES):]: ax.set_axis_off()
    ax = axes[-1]
    ax.imshow(montage(idxs), interpolation="nearest")
    shown = min(len(idxs), 16)
    ax.set_title(f"batch: {len(idxs)} images" + (f"  ({shown} shown)" if len(idxs) > shown else ""), fontsize=9)
    grid_n = next(iter(Zs.values())).shape[0]
    fig.suptitle(f"N = {n}    img seed {img_seed}    dir seed {dir_seed}    grid {grid_n}    extent {extent}    cap {loss_cap:g}",
                 fontsize=12)
    fig.tight_layout()
    return fig

def contour_fig(Z, title, extent=EXTENT, loss_cap=None, figsize=(5.5, 4.6)):
    loss_cap = LOSS_CAP if loss_cap is None else loss_cap
    fig = Figure(figsize=figsize); FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)
    cs = draw_contour(ax, Z, extent, loss_cap)
    ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$\beta$")
    ax.set_title(title, fontsize=9)
    fig.colorbar(cs, ax=ax, shrink=0.9)
    fig.tight_layout()
    return fig

def save_texture(Z, path, loss_cap=None, dpi=300, size_in=4):
    """Square, axis-free image for a manim texture. Orientation: imshow(rot90(Z.T)) -> alpha right, beta up (matches contour)."""
    loss_cap = LOSS_CAP if loss_cap is None else loss_cap
    fig = Figure(figsize=(size_in, size_in), frameon=False); FigureCanvasAgg(fig)
    ax = fig.add_axes([0., 0., 1., 1.]); ax.set_axis_off()
    ax.imshow(np.rot90(np.clip(Z, None, loss_cap).T), cmap=CMAP, interpolation="nearest")
    fig.savefig(path, bbox_inches="tight", pad_inches=0, dpi=dpi)


## Section 1: low-res exploration (32×32)

For every `(N, img seed, dir seed)`: draw N random val images, compute all 11 landscapes on that batch, save one 3×4 figure.
Raw losses for the figure are cached in a small `.npz` next to it (so the sweep resumes and figures can be re-rendered
with a different cap) — no per-landscape npy/textures here.

In [7]:
# ---- timing estimate: batch forward at the largest N, per model ----
x_big, y_big = load_batch(batch_indices(N_IMAGES_LIST[-1], IMG_SEEDS[0]))
total_low = 0.0
for mn in sorted({m for m, _ in LANDSCAPES.values()}):
    model = get_model(mn)
    n_cfg = sum(1 for m, _ in LANDSCAPES.values() if m == mn)
    with torch.no_grad():
        for _ in range(3): model(x_big)
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(10): crit(model(x_big), y_big).item()
        torch.cuda.synchronize(); dt = (time.time() - t0) / 10
    # per model: n_cfg configs x (sum over N of ~N/128 * dt, but small N is latency-bound) -- rough: all N at full cost
    per_fig = n_cfg * dt * GRID_LOW**2
    total_low += per_fig * len(IMG_SEEDS) * len(DIR_SEEDS) * len(N_IMAGES_LIST)
    print(f"{mn:9s} N={N_IMAGES_LIST[-1]}: {dt*1e3:6.1f} ms/eval   {n_cfg} cfgs   "
          f"~{per_fig/60:.1f} min per N={N_IMAGES_LIST[-1]} figure   hi-res {GRID_HIGH}^2 ~ {dt*GRID_HIGH**2/60:.0f} min per landscape")
print(f"\nupper bound for the whole low-res sweep (treating every N as N={N_IMAGES_LIST[-1]}): ~{total_low/3600:.1f} h")
del x_big, y_big


plain26   N=1:    1.8 ms/eval   1 cfgs   ~0.0 min per N=1 figure   hi-res 192^2 ~ 1 min per landscape
plain74   N=1:    2.6 ms/eval   4 cfgs   ~0.2 min per N=1 figure   hi-res 192^2 ~ 2 min per landscape
plain8    N=1:    0.3 ms/eval   2 cfgs   ~0.0 min per N=1 figure   hi-res 192^2 ~ 0 min per landscape
resnet74  N=1:    3.0 ms/eval   4 cfgs   ~0.2 min per N=1 figure   hi-res 192^2 ~ 2 min per landscape

upper bound for the whole low-res sweep (treating every N as N=1): ~1.8 h


In [8]:
# ---- SWEEP: one figure per (N, img seed, dir seed), all 11 landscapes each ----
def lowres_stem(n, img_seed, dir_seed):
    return f"n{n:03d}_imgseed{img_seed}_dir{dir_seed}_grid{GRID_LOW}"

def render_lowres(n, img_seed, dir_seed, loss_cap=None, show_inline=False):
    """Re-render a figure from its cached .npz (e.g. with a different loss_cap). Returns the png path."""
    stem = lowres_stem(n, img_seed, dir_seed)
    d = np.load(LOWRES_OUT / f"{stem}.npz")
    Zs = {cfg: d[cfg] for cfg in LANDSCAPES}
    cap = LOSS_CAP if loss_cap is None else loss_cap
    png = LOWRES_OUT / (f"{stem}.png" if cap == LOSS_CAP else f"{stem}_cap{cap:g}.png")
    grid_fig(Zs, d["idxs"], n, img_seed, dir_seed, EXTENT, cap).savefig(png, dpi=110)
    if show_inline: display(Image.open(png))
    return png

jobs = [(n, s, d) for n in N_IMAGES_LIST for s in IMG_SEEDS for d in DIR_SEEDS]
for n, s, d in (pbar := tqdm(jobs, desc="figures")):
    pbar.set_postfix_str(f"N={n} img{s} dir{d}")
    stem = lowres_stem(n, s, d)
    if SKIP_EXISTING and (LOWRES_OUT / f"{stem}.npz").exists():
        continue
    idxs = batch_indices(n, s)
    x, y = load_batch(idxs)
    Zs = {}
    for cfg, (mn, li) in LANDSCAPES.items():
        model = get_model(mn)
        delta, eta = make_directions(model, li, d)
        Zs[cfg] = loss_surface(model, delta, eta, x, y, GRID_LOW)
    np.savez(LOWRES_OUT / f"{stem}.npz", idxs=idxs, **Zs)
    render_lowres(n, s, d)
    del x, y
print("done ->", LOWRES_OUT)


figures:   6%|▉             | 16/256 [06:58<1:44:30, 26.13s/it, N=1 img55 dir24]


KeyboardInterrupt: 

In [ ]:
# browse, e.g.:
# render_lowres(32, 52, 25, show_inline=True)
# render_lowres(32, 52, 25, loss_cap=30, show_inline=True)
# for n in N_IMAGES_LIST: render_lowres(n, 52, 25, show_inline=True)   # walk through N for one seed pair


## Section 2: hi-res renders of the 11 landscapes

`FAVORITE` is the global pick of `(n_images, img_seed, dir_seed)`; `OVERRIDES` adjusts any of
`grid`, `extent`, `loss_cap`, `n_images`, `img_seed`, `dir_seed` per config. `CFGS` selects which landscapes to render.
Each landscape saves `.npy` (raw), `_contour.png`, `_tex.png`, and `.json` (incl. the batch's image indices) to `HIRES_OUT/<cfg>/`.
`loss_cap` only affects the pngs; re-running with a new cap on an existing `.npy` just rewrites the pngs (tagged `_cap<N>`).

In [ ]:
FAVORITE  = dict(n_images=32, img_seed=52, dir_seed=25, grid=512)
OVERRIDES = {
    "plain8_first3":   dict(extent=0.75),
    "plain8_last3":    dict(extent=0.75),
    "plain26_first4":  dict(extent=1.5),
    # "plain74_first8":  dict(grid=300, extent=1.5, loss_cap=64),
    # "resnet74_last8":  dict(dir_seed=26, n_images=64),
}
CFGS      = list(LANDSCAPES)          # or a subset, e.g. ["plain74_last8", "resnet74_last8"]
HIRES_OUT = OUT / "hires_v2"


In [ ]:
# ---- hi-res saving ----
def hires_paths(cfg, n, img_seed, dir_seed, grid_n, root, extent=EXTENT, loss_cap=None):
    """npy/json depend on (cfg, n, img_seed, dir_seed, grid, extent); pngs additionally get a _cap tag when loss_cap != LOSS_CAP."""
    d = root / cfg
    d.mkdir(parents=True, exist_ok=True)
    stem = f"{cfg}_n{n:03d}_imgseed{img_seed}_dir{dir_seed}_grid{grid_n}"
    if extent != EXTENT:
        stem += f"_ext{extent:g}"
    pstem = stem if (loss_cap is None or loss_cap == LOSS_CAP) else f"{stem}_cap{loss_cap:g}"
    return {"npy": d / f"{stem}.npy", "contour": d / f"{pstem}_contour.png",
            "tex": d / f"{pstem}_tex.png", "meta": d / f"{stem}.json"}

def hires_title(cfg, n, img_seed, dir_seed, Z, loss0, loss_cap):
    return (f"{cfg}   N={n}   img seed {img_seed}   dir seed {dir_seed}\n"
            f"min {Z.min():.2f}   max {Z.max():.1f}   L(0,0) {loss0:.2f}   cap {loss_cap:g}")

def write_pngs(Z, p, title, extent, loss_cap):
    contour_fig(Z, title, extent, loss_cap).savefig(p["contour"], dpi=150)
    save_texture(Z, p["tex"], loss_cap)

def rerender_hires(cfg, n, img_seed, dir_seed, grid_n, root, extent=EXTENT, loss_cap=None):
    """Rewrite contour + texture from a saved .npy with a different loss_cap. ~1 s, no model needed."""
    loss_cap = LOSS_CAP if loss_cap is None else loss_cap
    p = hires_paths(cfg, n, img_seed, dir_seed, grid_n, root, extent, loss_cap)
    Z = np.load(p["npy"])
    loss0 = json.loads(p["meta"].read_text())["loss_origin"]
    write_pngs(Z, p, hires_title(cfg, n, img_seed, dir_seed, Z, loss0, loss_cap), extent, loss_cap)
    return Z

def run_hires(cfg, n, img_seed, dir_seed, grid_n, root, extent=EXTENT, loss_cap=None):
    """Compute + save one hi-res landscape. If its .npy exists (and SKIP_EXISTING), only the pngs are (re)written."""
    loss_cap = LOSS_CAP if loss_cap is None else loss_cap
    p = hires_paths(cfg, n, img_seed, dir_seed, grid_n, root, extent, loss_cap)
    if SKIP_EXISTING and p["npy"].exists():
        return rerender_hires(cfg, n, img_seed, dir_seed, grid_n, root, extent, loss_cap)
    mn, li = LANDSCAPES[cfg]
    model = get_model(mn)
    idxs = batch_indices(n, img_seed)
    x, y = load_batch(idxs)
    with torch.no_grad():
        loss0 = crit(model(x), y).item()
    delta, eta = make_directions(model, li, dir_seed)
    t0 = time.time()
    Z = loss_surface(model, delta, eta, x, y, grid_n, extent)
    elapsed = time.time() - t0
    np.save(p["npy"], Z)
    write_pngs(Z, p, hires_title(cfg, n, img_seed, dir_seed, Z, loss0, loss_cap), extent, loss_cap)
    names = weighted_layers(model)
    meta = dict(cfg=cfg, model=mn, layer_idxs=li, layer_names=[names[i][0] for i in li],
                n_images=int(n), img_seed=int(img_seed), dir_seed=int(dir_seed),
                img_idxs=[int(i) for i in idxs], wnids=[val_ds.classes[val_ds.targets[int(i)]] for i in idxs],
                grid=int(grid_n), extent=float(extent), loss_cap=float(loss_cap),
                loss_min=float(Z.min()), loss_max=float(Z.max()), loss_origin=float(loss0),
                seconds=round(elapsed, 1),
                layout="Z[i,j] = mean batch loss at alpha=lin[j], beta=lin[i]; texture = imshow(rot90(Z.T))")
    p["meta"].write_text(json.dumps(meta, indent=2))
    del x, y
    return Z

def show_hires(cfg, n, img_seed, dir_seed, grid_n, root, extent=EXTENT, loss_cap=None):
    display(Image.open(hires_paths(cfg, n, img_seed, dir_seed, grid_n, root, extent, loss_cap)["contour"]))


In [ ]:
# ---- render ----
DEFAULTS = dict(grid=GRID_HIGH, extent=EXTENT, loss_cap=LOSS_CAP)
specs = {cfg: {**DEFAULTS, **FAVORITE, **OVERRIDES.get(cfg, {})} for cfg in CFGS}
for cfg, sp in specs.items():
    print(f"{cfg:16s} N={sp['n_images']:3d}  img{sp['img_seed']}  dir{sp['dir_seed']}  grid {sp['grid']}  extent {sp['extent']}  cap {sp['loss_cap']:g}")

for cfg, sp in tqdm(specs.items(), desc="hi-res"):
    args = (cfg, sp["n_images"], sp["img_seed"], sp["dir_seed"], sp["grid"], HIRES_OUT, sp["extent"], sp["loss_cap"])
    run_hires(*args)
    show_hires(*args)
print("done ->", HIRES_OUT)
